In [1]:
import os
import sys
from pathlib import Path
from multiprocessing import Pool, cpu_count

import numpy as np
import pandas as pd
import torch
import rpy2.robjects as robjects
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

torch.manual_seed(756)
np.random.seed(756)

# =========================================================
# Notebook mode
# Current notebook assumed at:
#   /MFDNN_code/EEG
# =========================================================
EEG_DIR = Path.cwd().resolve()
PROJECT_ROOT = EEG_DIR.parent
DATA_DIR = EEG_DIR / "Data"
RESULTS_DIR = EEG_DIR / "Results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

from Method.mfdnn import *
from Method.utils import *

train_indices_list = pd.read_csv(DATA_DIR / "train_indices_list.csv")
test_indices_list = pd.read_csv(DATA_DIR / "test_indices_list.csv")

rds_file = DATA_DIR / "EEG_y.rds"
r_array = robjects.r["readRDS"](str(rds_file))

EEG_x = np.array(r_array)
EEG_y = np.array([1] * 39 + [-1] * 22)

n = 61
p = 1
frun = 50
# frun = 50
domain_range = [np.array([0, 1]), np.array([1, 64])]

model_params = {
    "num_basis": [5, 5],
    "layer_sizes": [64, 64],
    "epochs": 200,
    "val_ratio": 0.25,
    "patience": 10,
    "basis_type": "bspline",
    "degree": 3,
}

lam1_values = [0.01, 0.05, 0.1, 0.5, 1, 2, 5]
lam2_values = [0, 0.001, 0.01, 0.1, 1, 2, 5]

In [2]:
class ClassificationNN(nn.Module):
    def __init__(self, input_size, hidden_layer_sizes):
        super(ClassificationNN, self).__init__()
        layers = []

        self.input = nn.Linear(input_size, hidden_layer_sizes[0])
        layers.append(self.input)
        layers.append(nn.ReLU())

        for i in range(len(hidden_layer_sizes) - 1):
            layers.append(nn.Linear(hidden_layer_sizes[i], hidden_layer_sizes[i + 1]))
            layers.append(nn.ReLU())

        layers.append(nn.Linear(hidden_layer_sizes[-1], 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.model(x)

    @property
    def input_weight(self):
        return self.input.weight

In [3]:
def MFDNN_classification(
    p,
    resp,
    func_cov,
    num_basis,
    layer_sizes,
    domain_range,
    epochs,
    val_ratio,
    patience,
    lam1=0,
    lam2=0,
    epsilon=0.001,
    std_resp=True,
    basis_type="bspline",
    degree=3,
):
    A = integral(
        func_cov,
        num_basis,
        domain_range,
        basis_type=basis_type,
        degree=degree,
    )
    S = smooth_penalty(
        func_cov,
        num_basis,
        domain_range,
        basis_type=basis_type,
        degree=degree,
    )
    S = torch.tensor(S, dtype=torch.float32)
    N = A.shape[0]

    if len(A.shape) == 2:
        M = A.shape[1]
        input_size = M
    else:
        M = A.shape[2]
        input_size = p * M

    resp01 = np.where(np.asarray(resp).reshape(-1) > 0, 1.0, 0.0)

    if val_ratio is not None:
        trainX, validationX, trainy, validationy = train_test_split(
            A, resp01, test_size=val_ratio, random_state=42
        )
        trainX = torch.tensor(trainX).float()
        trainy = torch.tensor(trainy).view(-1, 1).float()
        validationX = torch.tensor(validationX).float()
        validationy = torch.tensor(validationy).view(-1, 1).float()
    else:
        trainX = torch.tensor(A).float()
        trainy = torch.tensor(resp01).view(N, 1).float()

    model = ClassificationNN(input_size, layer_sizes)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    train_losses = []
    validation_losses = []

    if patience is not None:
        best_val_loss = float("inf")
        stopping_patience = patience

    for _ in range(epochs):
        optimizer.zero_grad()
        outputs = model(trainX)
        loss = criterion(outputs, trainy)

        if lam1 == 0 or p == 1:
            weight = model.input_weight
            norm = torch.sum((weight @ S) * weight, dim=1)
            regularization = lam2 * torch.mean(norm)
            total_loss = loss + regularization
            total_loss.backward()
        else:
            weight = model.input_weight
            norm1 = torch.sum(weight ** 2, dim=0).view(p, M).sum(dim=1)
            norm2 = torch.sum((weight @ S) * weight, dim=1)
            regularization1 = lam1 * torch.sum(torch.sqrt(norm1 + 1e-6))
            regularization2 = lam2 * torch.mean(norm2)
            total_loss = loss + regularization1 + regularization2
            total_loss.backward()

        optimizer.step()
        train_losses.append(loss.item())

        if patience is not None:
            val_loss = criterion(model(validationX), validationy)
            validation_losses.append(val_loss.item())

            if val_loss < best_val_loss and best_val_loss - val_loss >= epsilon:
                best_val_loss = val_loss
                stopping_patience = patience
            else:
                stopping_patience -= 1
                if stopping_patience == 0:
                    break

    if p == 1:
        l21_norm = torch.sqrt(torch.sum(model.input_weight ** 2, dim=0))
    else:
        l21_norm = torch.sqrt(torch.sum(model.input_weight ** 2, dim=0).view(p, M).sum(dim=1))

    return train_losses, validation_losses, model, l21_norm


def MFDNN_predict_classification(
    p,
    model,
    func_cov,
    num_basis,
    domain_range,
    basis_type="bspline",
    degree=3,
):
    A = integral(
        func_cov,
        num_basis,
        domain_range,
        basis_type=basis_type,
        degree=degree,
    )
    testX = torch.tensor(A).float()
    logits = model(testX)
    return logits

In [4]:
def select_best_hyperparameters(
    X_train,
    y_train,
    p,
    domain_range,
    lam1_values,
    lam2_values,
    model_params,
):
    mse_results = np.zeros((len(lam1_values), len(lam2_values)))
    model_info = {}

    for i, lam1 in enumerate(lam1_values):
        for j, lam2 in enumerate(lam2_values):
            try:
                train_losses, val_losses, model, _ = MFDNN_classification(
                    p=p,
                    resp=y_train,
                    func_cov=X_train,
                    num_basis=model_params["num_basis"],
                    layer_sizes=model_params["layer_sizes"],
                    domain_range=domain_range,
                    epochs=model_params["epochs"],
                    val_ratio=model_params["val_ratio"],
                    patience=model_params["patience"],
                    lam1=lam1,
                    lam2=lam2,
                    std_resp=True,
                    basis_type=model_params["basis_type"],
                    degree=model_params["degree"],
                )

                mse = min(val_losses) if len(val_losses) > 0 else np.mean(train_losses[-10:])
                mse_results[i, j] = mse
                model_info[f"{i}_{j}"] = model

            except Exception:
                mse_results[i, j] = np.inf
                model_info[f"{i}_{j}"] = None

    best_idx = np.unravel_index(np.argmin(mse_results), mse_results.shape)
    best_lam1 = lam1_values[best_idx[0]]
    best_lam2 = lam2_values[best_idx[1]]
    best_model = model_info[f"{best_idx[0]}_{best_idx[1]}"]

    return best_lam1, best_lam2, best_model

In [5]:
def evaluate_on_test_set(best_model, X_test, y_test, p, domain_range, model_params):
    try:
        logits = MFDNN_predict_classification(
            p,
            best_model,
            X_test,
            model_params["num_basis"],
            domain_range,
            basis_type=model_params["basis_type"],
            degree=model_params["degree"],
        ).detach().numpy()

        test_labels = np.where(logits.flatten() > 0, 1, -1)
        f1 = f1_score(y_test, test_labels, pos_label=1)

        return f1, test_labels
    except Exception:
        return 0.0, None

# MFDNN_class

In [6]:
all_f1 = []
all_best_lam1 = []
all_best_lam2 = []

for run_idx in range(frun):
    print(f"Run {run_idx + 1}/{frun}")

    train_indices = train_indices_list.iloc[:, run_idx].to_numpy() - 1
    test_indices = test_indices_list.iloc[:, run_idx].to_numpy() - 1

    X_train = np.transpose(EEG_x[:, :, train_indices], (2, 0, 1))
    X_test = np.transpose(EEG_x[:, :, test_indices], (2, 0, 1))
    y_train = EEG_y[train_indices]
    y_test = EEG_y[test_indices]

    best_lam1, best_lam2, best_model = select_best_hyperparameters(
        X_train,
        y_train,
        p,
        domain_range,
        lam1_values,
        lam2_values,
        model_params,
    )

    f1, _ = evaluate_on_test_set(
        best_model,
        X_test,
        y_test,
        p,
        domain_range,
        model_params,
    )

    all_f1.append(f1)
    all_best_lam1.append(best_lam1)
    all_best_lam2.append(best_lam2)

mean_f1 = np.mean(all_f1)
std_f1 = np.std(all_f1)

print("\nSummary")
print(f"Mean F1: {mean_f1:.6f}")
print(f"Std F1:  {std_f1:.6f}")

Run 1/50
Run 2/50
Run 3/50
Run 4/50
Run 5/50
Run 6/50
Run 7/50
Run 8/50
Run 9/50
Run 10/50
Run 11/50
Run 12/50
Run 13/50
Run 14/50
Run 15/50
Run 16/50
Run 17/50
Run 18/50
Run 19/50
Run 20/50
Run 21/50
Run 22/50
Run 23/50
Run 24/50
Run 25/50
Run 26/50
Run 27/50
Run 28/50
Run 29/50
Run 30/50
Run 31/50
Run 32/50
Run 33/50
Run 34/50
Run 35/50
Run 36/50
Run 37/50
Run 38/50
Run 39/50
Run 40/50
Run 41/50
Run 42/50
Run 43/50
Run 44/50
Run 45/50
Run 46/50
Run 47/50
Run 48/50
Run 49/50
Run 50/50

Summary
Mean F1: 0.817472
Std F1:  0.079188


# MFDNN

In [7]:
# =========================================================
# Original MFDNN on EEG
# =========================================================
def select_best_hyperparameters_original(
    X_train,
    y_train,
    p,
    domain_range,
    lam1_values,
    lam2_values,
    model_params,
):
    mse_results = np.zeros((len(lam1_values), len(lam2_values)))
    model_info = {}

    for i, lam1 in enumerate(lam1_values):
        for j, lam2 in enumerate(lam2_values):
            try:
                train_losses, val_losses, model, _ = MFDNN(
                    p=p,
                    resp=y_train,
                    func_cov=X_train,
                    num_basis=model_params["num_basis"],
                    layer_sizes=model_params["layer_sizes"],
                    domain_range=domain_range,
                    epochs=model_params["epochs"],
                    val_ratio=model_params["val_ratio"],
                    patience=model_params["patience"],
                    lam1=lam1,
                    lam2=lam2,
                    std_resp=True,
                    basis_type=model_params["basis_type"],
                    degree=model_params["degree"],
                )

                mse = min(val_losses) if len(val_losses) > 0 else np.mean(train_losses[-10:])
                mse_results[i, j] = mse
                model_info[f"{i}_{j}"] = model

            except Exception:
                mse_results[i, j] = np.inf
                model_info[f"{i}_{j}"] = None

    best_idx = np.unravel_index(np.argmin(mse_results), mse_results.shape)
    best_lam1 = lam1_values[best_idx[0]]
    best_lam2 = lam2_values[best_idx[1]]
    best_model = model_info[f"{best_idx[0]}_{best_idx[1]}"]

    return best_lam1, best_lam2, best_model


def evaluate_on_test_set_original(best_model, X_test, y_test, p, domain_range, model_params):
    try:
        pred = MFDNN_predict(
            p,
            best_model,
            X_test,
            model_params["num_basis"],
            domain_range,
            basis_type=model_params["basis_type"],
            degree=model_params["degree"],
        ).detach().numpy()

        test_labels = np.where(pred.flatten() > 0, 1, -1)
        f1 = f1_score(y_test, test_labels, pos_label=1)

        return f1, test_labels
    except Exception:
        return 0.0, None


all_f1_original = []
all_best_lam1_original = []
all_best_lam2_original = []

for run_idx in range(frun):
    print(f"Original MFDNN Run {run_idx + 1}/{frun}")

    train_indices = train_indices_list.iloc[:, run_idx].to_numpy() - 1
    test_indices = test_indices_list.iloc[:, run_idx].to_numpy() - 1

    X_train = np.transpose(EEG_x[:, :, train_indices], (2, 0, 1))
    X_test = np.transpose(EEG_x[:, :, test_indices], (2, 0, 1))
    y_train = EEG_y[train_indices]
    y_test = EEG_y[test_indices]

    best_lam1, best_lam2, best_model = select_best_hyperparameters_original(
        X_train,
        y_train,
        p,
        domain_range,
        lam1_values,
        lam2_values,
        model_params,
    )

    f1, _ = evaluate_on_test_set_original(
        best_model,
        X_test,
        y_test,
        p,
        domain_range,
        model_params,
    )

    all_f1_original.append(f1)
    all_best_lam1_original.append(best_lam1)
    all_best_lam2_original.append(best_lam2)

mean_f1_original = np.mean(all_f1_original)
std_f1_original = np.std(all_f1_original)

print("\nOriginal MFDNN summary")
print(f"Mean F1: {mean_f1_original:.6f}")
print(f"Std F1:  {std_f1_original:.6f}")

Original MFDNN Run 1/50
Original MFDNN Run 2/50
Original MFDNN Run 3/50
Original MFDNN Run 4/50
Original MFDNN Run 5/50
Original MFDNN Run 6/50
Original MFDNN Run 7/50
Original MFDNN Run 8/50
Original MFDNN Run 9/50
Original MFDNN Run 10/50
Original MFDNN Run 11/50
Original MFDNN Run 12/50
Original MFDNN Run 13/50
Original MFDNN Run 14/50
Original MFDNN Run 15/50
Original MFDNN Run 16/50
Original MFDNN Run 17/50
Original MFDNN Run 18/50
Original MFDNN Run 19/50
Original MFDNN Run 20/50
Original MFDNN Run 21/50
Original MFDNN Run 22/50
Original MFDNN Run 23/50
Original MFDNN Run 24/50
Original MFDNN Run 25/50
Original MFDNN Run 26/50
Original MFDNN Run 27/50
Original MFDNN Run 28/50
Original MFDNN Run 29/50
Original MFDNN Run 30/50
Original MFDNN Run 31/50
Original MFDNN Run 32/50
Original MFDNN Run 33/50
Original MFDNN Run 34/50
Original MFDNN Run 35/50
Original MFDNN Run 36/50
Original MFDNN Run 37/50
Original MFDNN Run 38/50
Original MFDNN Run 39/50
Original MFDNN Run 40/50
Original 

In [8]:
df_compare = pd.DataFrame(
    {
        "method": ["MFDNN_classification", "MFDNN_original"],
        "mean_f1": [mean_f1, mean_f1_original],
        "std_f1": [std_f1, std_f1_original],
    }
)

save_compare = RESULTS_DIR / "EEG_MFDNN.csv"
df_compare.to_csv(save_compare, index=False, encoding="utf-8-sig")

print("\nComparison")
print(df_compare)
print(f"\nSaved to: {save_compare}")


Comparison
                 method   mean_f1    std_f1
0  MFDNN_classification  0.817472  0.079188
1        MFDNN_original  0.745833  0.094872

Saved to: /Users/wangdongxue/Desktop/MFDNN_STCO_revise/MFDNN_code/EEG/Results/EEG_MFDNN.csv
